*GRANGER M., LEVERS G. - 3A ESTACA*

# **Projet PIRATE 3A : Impact potentiel du Machine Learnung dans le traitement de données d'essais mécanique pour l'optimisation de structures**

L'objectif de ce Notebook est de tester différents modèles de ML pour de la classification d'éprouvettes en fonction de leur module de flexion calculé à partir d'un essai de flexion en 3 points (Norme NF EN ISO 14125).

Les modèles qui seront testés sont les suivants :
* SVR
* k-NN
* Random Forest


---


## **Traitement des données d'essais**

### **Chargement des données**

Les fichiers de données `.csv` utilisés ont été générés artificiellement par un scripte python suivant la norme *NF EN ISO 14125*, en y introduisant du bruit.

Nous chargeons les données de simulation vers un dataframe python pour pouvoir les analyser, puis les manipuler.

In [3]:
import pandas as pd

df = pd.read_csv('data/simulation_flexion3pts_final.csv')
print('Données chargées.')

Données chargées.


### **Analyse des données**

Nous choisissons d'afficher les valeurs des modules d'Young de chaque éprouvette afin d'avoir une idée de la répartition graphique.

Afin de s'intéresser à la densité de nos points dans certaines zones, nous choisissons un diagramme en violon.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#--- Répartition graphique ---
plt.figure(figsize=(7, 4.5))
plt.scatter(df.index, df['Module E'])
plt.title('Répartition des valeurs de Module E')
plt.ylabel('Module E (MPa)')
plt.show()

#--- Diagramme en violon (densité) ---
plt.figure(figsize=(7, 4.5))
sns.violinplot(y=df['Module E'], color='#BB87FF')
plt.title('Diagramme en Violon')
plt.ylabel('Module E (MPa)')
plt.show()

Comme nous le voyons sur le premier graphique, et illustré sur le second, il y a trois zones distinctes de module d'Young.


1.   Pic du haut : zone où les éprouvette seront considérées comme **parfaites**.

2.   Pic du milieu : zone où les éprouvettes seront considérées comme **acceptables**.

3.   Pic du bas : zone où les éprouvettes seront considérées comme **à jeter**.

### **Analyse du dataframe**

Notre dataframe contient différentes variables provenant des caractéristiques géométriques de l'éprouvette, des conditions de l'essais et des efforts appliqués.

In [ ]:
#--- Informations sur les variables ---
print('Informations sur les variables :')
df.info()

#--- Description du dataframe ---
print('\nDescription du dataframe :')
display(df.describe())

#--- Affichage des premières et dernières lignes du dataframe ---
print('\nAffichage des premières lignes du dataframe:')
display(df.head())
print('\nAffichage des dernières lignes du dataframe:')
display(df.tail())

## **Valeur de référence et seuils**

La valeur de référence utilisée pour la construction des données d'essais est un module d'Young de valeur $E = 210\ 000 \text{MPa}$ (correspondant au module d'Young d'un acier). Nous prendrons donc cette valeur pour référence.

In [ ]:
module_ref = 210000

Pour la suite nous définissons des seuils pour la classification de nos modules d'Young :

*   $0.80 \times E_{ref} < E$ : l'éprouvette est classé dans la catégorie "**Parfait**".

*   $0.50 \times E_{ref} < E < 0.80 \times E_{ref}$ : l'éprouvette est classée dans la catégorie "**Acceptable**".

*   $E < 0.50 \times E_{ref}$ : l'éprouvette est classée dans la catégorie "**À jeter**".

Nous créons ainsi une fonction d'étiquetage pour classer les modules de test, et par la suite les modules qui seront prédis.

In [ ]:
#--- Création de la fonction d'étiquetage ---
def etiquetage(module) :

  ratio = module/module_ref

  if ratio >= 0.8 :
    return 'Parfait'

  elif 0.5 <= ratio < 0.8 :
    return 'Acceptable'

  else :
    return 'A jeter'

---

## **Choix des features et de la target**

### **Matrice de corrélation**

Notre dataframe possède de nombreuses variables, afin d'avoir une idée des meilleures à choisir comme *features* nous affichons la matrice de corrélation.




In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

### **Définition des features**

Notre objectif ici est de permettre à l'utilisateur de réduire le nombre de variables nécessaires pour prédire un module d'Young avec précision.

Dans cette démarches les *features* retenues sont les suivantes :

*   Largeur de l'éprouvette
*   Épaisseur de l'éprouvette
*   Distance inter-appui
*   Delta2 (flèche)
*   Effort2 (effort appliqué)


In [ ]:
X = df[['Largeur eprouvette',
        'Epaisseur',
        'Distance inter-appui',
        'delta2',
        'effort2']]

### **Définition de la target**

Notre objectif est de prédire le module d'Young de l'éprouvette, c'est donc notre target.

In [ ]:
Y = df['Module E']

---
## **Définition des conditions d'entrainement**

### **Séparation entrainement/test**

Nous choisissons ici de donner 70% de nos données au modèle pour qu'il puisse s'entrainer, et 30% pour qu'il puisse se tester.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)

### **Création des pipelines**

Le pipeline permet de garantir que la normalisation soit ajustée uniquement sur les plis d'entrainement pendant la validation croisée.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

# --- SVR ---
pipeline_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

#--- KNN ---
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

#--- RF ---
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

### **Définition de la grille d'hyperparamètres**

Dans notre objectif de trouver le meilleur modèle de ML pour notre cas, sur nos données, nous allons réaliser un `GridSearch` afin d'optimiser les paramètres du modèle. Nous définissons au préalable la grille des hyperparamètres à explorer et associer.



In [ ]:
#--- SVR ---
param_grid_svr = {
    'svr__kernel': ['linear', 'rbf', 'poly'],
    'svr__C': [1000.0, 10000.0, 100000.0],           # Paramètre de régularisation
    'svr__epsilon': [0.0001, 0.001, 0.01],   # Marge de tolérance (tube)
    'svr__gamma': ['scale', 'auto', 0.1, 0.5, 1] # Coefficient pour les noyaux rbf/poly
}

#--- KNN ---
param_grid_knn = {
    'knn__n_neighbors': [2, 3, 4, 5, 7, 10, 15, 20, 30, 50],      # On va chercher un peu plus loin
    'knn__weights': ['uniform', 'distance'],            # Parfait tel quel
    'knn__p': [1, 2],                                   # Parfait tel quel
    'knn__algorithm': ['auto', 'ball_tree', 'kd_tree'], # On teste les arbres
    'knn__leaf_size': [10, 20, 30, 40]                      # Quelques tailles de feuilles
}

#--- RF ---
param_grid_rf = {
    'rf__n_estimators': [100, 200, 300, 500, 1000],          # 3 tailles de forêt
    'rf__max_depth': [None, 5, 10, 14, 15, 16, 20],              # On teste sans limite, et avec des limites
    'rf__min_samples_split': [2, 3, 5],              # On teste le défaut et un élagage léger
    'rf__min_samples_leaf': [1, 2, 3],               # Idem pour les feuilles finales
    'rf__max_features': ['sqrt', None]            # Un test avec toutes les colonnes, un test avec une racine carrée
}

---
## **Configuration et lancement de la recherche en grille**

Nous configurons la recherche en grille à l'aide du pipeline et de la grille d'hyperparamères, en choisissant une validation croisée à 5 plis et de minimiser l'erreur quadratique moyenne.

### **GridSearch : Support Vector Regression**

In [ ]:
from sklearn.model_selection import GridSearchCV

#--- Configuration ---
gridsearch_svr = GridSearchCV(
    estimator=pipeline_svr,
    param_grid=param_grid_svr,
    cv=5,                              # Validation croisée à 5 plis
    scoring='neg_mean_squared_error',  # Optimiser pour réduire l'erreur quadratique moyenne
    n_jobs=-1,                         # Utiliser tous les cœurs du processeur
    verbose=1                          # Afficher la progression
)

#--- Lancement du GridSearch ---
print('Début GridSearch - SVR')
gridsearch_svr.fit(x_train, y_train)
print('Fin GridSearch - SVR')

#--- Affichage des résultats ---
print('\n--- Meilleur SVR ---')
print(f'Meilleurs paramètres : {gridsearch_svr.best_params_}')

### **GridSearch : K-Nearest Neighbors**

In [ ]:
#--- Configuration ---
gridsearch_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid_knn,
    cv=5,                              # Validation croisée à 5 plis
    scoring='neg_mean_squared_error',  # Optimiser pour réduire l'erreur quadratique moyenne
    n_jobs=-1,                         # Utiliser tous les cœurs du processeur
    verbose=1                          # Afficher la progression
)

#--- Lancement du GridSearch ---
print('Début GridSearch - KNN')
gridsearch_knn.fit(x_train, y_train)
print('Fin GridSearch - KNN')

#--- Affichage des résultats ---
print('\n--- Meilleur KNN ---')
print(f'Meilleurs paramètres : {gridsearch_knn.best_params_}')

### **GridSearch : Random Forest**

In [ ]:
#--- Configuration ---
gridsearch_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=5,                              # Validation croisée à 5 plis
    scoring='neg_mean_squared_error',  # Optimiser pour réduire l'erreur quadratique moyenne
    n_jobs=-1,                         # Utiliser tous les cœurs du processeur
    verbose=1                          # Afficher la progression
)

#--- Lancement du GridSearch ---
print('Début GridSearch - RF')
gridsearch_rf.fit(x_train, y_train)
print('Fin GridSearch - RF')

#--- Affichage des résultats ---
print('\n--- Meilleur RF ---')
print(f'Meilleurs paramètres : {gridsearch_rf.best_params_}')

## **Création des modèles et entrainement**

Une fois les meilleurs hyperparamètres obtenus, nous pouvons créer le meilleur modèle avec. Grâce à la fonction `GridSearch`, le modèle est automatiquement réentrainé sur l'ensemble des données d'entrainement et prêt à l'emploi.



In [ ]:
#--- SVR ---
best_svr = gridsearch_svr.best_estimator_
y_pred_svr = best_svr.predict(x_test)

#--- KNN ---
best_knn = gridsearch_knn.best_estimator_
y_pred_knn = best_knn.predict(x_test)

#--- RF ---
best_rf = gridsearch_rf.best_estimator_
y_pred_rf = best_rf.predict(x_test)

---

## **Analyse des performances**

### **Calcul des performances**

Nous choisissons de calculer la racine de l'erreur quadratique moyenne ($\text{RMSE}$) définie par :
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

Nous choisissons aussi de calculer le coefficient de détermination ($\text{r²}$) défini par :
$$\text{r²}=1-\frac{ \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{N} (y_i - \bar{y})^2}$$


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

predictions = {
    "SVR": y_pred_svr,
    "KNN": y_pred_knn,
    "RF": y_pred_rf
}

for nom_modele, y_pred in predictions.items():
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- Modèle {nom_modele} ---")
    print(f"RMSE pour le {nom_modele} : {rmse:.4f}")
    print(f"R² pour le {nom_modele}   : {r2:.4f}")

### **Comparaison entre les prédictions et de la réalité**

Nous traçons sur un même graphique les valeurs réelles et les valeurs prédites afin de visualiser graphiquement les erreurs.

In [ ]:
for nom_modele, y_pred in predictions.items():
    plt.figure(figsize=(7, 4.5))

    plt.scatter(y_test, y_pred, label=nom_modele, color='blue')

    # Ligne idéale y = x
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], color='black')

    #Affichage des différents seuils
    plt.axhline(0.8 * module_ref, linestyle='--', label="80% de ref", color='black')
    plt.axhline(0.5 * module_ref, linestyle='--', label="50% de ref", color='black')

    plt.xlabel("Valeurs réelles")
    plt.ylabel("Valeurs prédites")
    plt.title(f"Comparaison valeurs réelles vs prédites - {nom_modele}")

    plt.legend()
    plt.show()



### **Matrice de confusion**

Afin de mesurer plus clairement les performances de nos modèles dans l'objectif de classifier des éprouvettes, nous reconstruisons une matrice de confusion manuellement.

In [ ]:
# --- Création des étiquettes pour les valeurs réelles ---
y_true_labels = [etiquetage(y) for y in y_test]

# --- Création des étiquettes pour les prédictions ---
predictions_labels = {}
for nom_modele, y_pred in predictions.items():
    y_pred_labels = [etiquetage(y) for y in y_pred]
    predictions_labels[nom_modele] = y_pred_labels

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

couleurs_modeles = {
    "SVR": "Blues",
    "KNN": "Purples",
    "RF": "Oranges"
}

#--- Matrices de confusion ---
for nom_modele, y_pred_labels in predictions_labels.items():
    plt.figure()
    cm = confusion_matrix(y_true_labels, y_pred_labels, labels=['A jeter', 'Acceptable', 'Parfait'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['A jeter', 'Acceptable', 'Parfait'])
    disp.plot(cmap=couleurs_modeles[nom_modele])
    plt.title(f"Matrice de confusion {nom_modele}")
    plt.show()


--- 
## **Exportation du meilleur modèle (SVR)**

In [ ]:
from sklearn.pipeline import make_pipeline
import joblib

# On sauvegarde ce pipeline unique dans un fichier
joblib.dump(best_svr, 'model_export/modele_svr_final.pkl')

print("Modèle exporté avec succès !")

Ainsi le modèle est exporté avec ses paramètres et ses poids, il est maintenant possible de le chargé sur un autre fichier et de l'utiliser pour prédire des valeurs de façcon fiable (vu les performances).

### **Vérification de la conformité de l'export**

Nous souhaitons vérifier que le modèle exporté contient bien le `Scaler()`.

Pour cela on charge notre modèle exporté.

In [ ]:
model_test = joblib.load(r'model_export/modele_svr_final.pkl')

print('Modèle chargé.')

On prédit une valeur à l'aide du modèle chargé.

In [ ]:
y_pred_test = model_test.predict(x_test.head(1))

print('--- Valeur prédite avant exporation ---')
print(y_pred_svr[0])

print('--- Valeur prédite après exporation ---')
print(y_pred_test[0])

if y_pred_svr[0] == y_pred_test[0] : 
    print('Exportation réussie avec Scaler().')


---
## **Explicabilité du modèle SVR avec SHAP**

La méthode **SHAP** (SHapley Additive exPlanations) permet de comprendre comment chaque variable influence la prédiction du modèle. 

Contrairement aux modèles basés sur des arbres, le SVR nécessite l'utilisation d'un `KernelExplainer`. Cette méthode étant coûteuse en temps de calcul, nous allons utiliser un échantillon réduit de nos données d'entraînement pour définir le comportement de base du modèle, et analyser un sous-ensemble de nos données de test.


In [ ]:
import shap

# Initialisation de l'affichage JavaScript dans le notebook (requis pour les graphiques SHAP)
shap.initjs()

# Échantillonnage des données d'entraînement pour réduire le temps de calcul (100 individus)
background_data = shap.sample(x_train, 100)

# SOLUTION : Utilisation d'une fonction lambda pour masquer le Pipeline à SHAP
explainer_svr = shap.KernelExplainer(lambda x_m: best_svr.predict(x_m), background_data)

# Calcul des valeurs SHAP sur un sous-échantillon du jeu de test (ex: les 30 premières éprouvettes)
x_test_sample = x_test[:30]
shap_values_svr = explainer_svr.shap_values(x_test_sample)


### **Explicabilité globale (Summary Plot)**

Le graphique suivant montre l'importance globale de chaque variable sur les prédictions du modèle SVR pour notre échantillon de test.


In [ ]:
# Affichage du Summary Plot
shap.summary_plot(shap_values_svr, x_test_sample, feature_names=X.columns)

On y remarque que par exemple, plus la valeur de F2 est élevée, plus la valeur du
module prédit sera grande. À l’inverse, plus l’épaisseur augmente plus elle tire la valeur
de la prédiction vers le bas.

### **Explicabilité locale (Force Plot)**

Regardons en détail pourquoi le modèle a fait cette prédiction spécifique pour la **première éprouvette** de notre échantillon de test. Les variables en rouge poussent la valeur prédite à la hausse, celles en bleu à la baisse.


In [ ]:
# Index de l'éprouvette à analyser
index_eprouvette = 0

valeur_reelle = y_test.iloc[index_eprouvette]
valeur_predite = best_svr.predict(x_test_sample.iloc[[index_eprouvette]])[0]

print(f"Analyse de l'éprouvette {index_eprouvette} :")
print(f"Valeur réelle = {valeur_reelle:.2f} MPa")
print(f"Prédiction = {valeur_predite:.2f} MPa")

# Affichage du Force Plot
shap.force_plot(
    base_value=explainer_svr.expected_value, 
    shap_values=shap_values_svr[index_eprouvette].round(3), 
    features=x_test_sample.iloc[index_eprouvette].round(3),
    feature_names=X.columns,
    matplotlib=True # Vous pouvez mettre False si l'affichage graphique classique vous gêne et que vous préférez la version web interactive
)


Sur ce force plot, nous pouvons observer que la valeur de base correspond à la moyenne des modules soit $\approx 125\ 000 \text{MPa}$. Ensuite la variable effort2 a fait monter la valeur de la prédiction, tandis que les autres en bleu l'ont fait diminuer légèrement, pour atteindre la valeur finale. 

--- 
## **Optimisation**

Avec la bibliotèque scipy il est possible d'optimiser notre meilleur modèle SVR en venant encore plus affiner les hyperparamètres pour obtenir un modèle encore plus performant. 

### **Définition du point de départ de l'optimisation**

Pour accélérer et faciliter la convergence de l'algorithme d'optimisation de Scipy nous allons repartir des meilleurs paramètres déterminés lors de notre précédent `GridSearch` comme point de départ (`x0`).

In [ ]:
from scipy.optimize import minimize
from sklearn.model_selection import cross_val_score
import numpy as np

# Récupération du point de départ depuis le GridSearch
best_C = gridsearch_svr.best_params_['svr__C']
best_eps = gridsearch_svr.best_params_['svr__epsilon']
best_gamma = gridsearch_svr.best_params_['svr__gamma']

# Si gamma est une chaîne de caractères (ex: 'scale'), on force une valeur numérique
if isinstance(best_gamma, str):
    best_gamma = 0.1

# Définition du point de départ x0 pour l'optimisation
x0 = [best_C, best_eps, best_gamma]

print(f"Point de départ (GridSearch) : C={x0[0]}, epsilon={x0[1]}, gamma={x0[2]}")


### **Définition de la fonction objectif**

Notre objectif pour notre modèle est de minimiser l'erreur quadratique moyenne, ainsi on crée une fonction objectif qui retourne l'erreur quratique moyenne.

In [ ]:
def objective_svr(params):
    C, epsilon, gamma = params
    
    # Mise à jour des paramètres du pipeline SVR existant
    pipeline_svr.set_params(
        svr__C=C,
        svr__epsilon=epsilon,
        svr__gamma=gamma
    )
    
    # Calcul du RMSE moyen par validation croisée
    scores = cross_val_score(
        pipeline_svr, 
        x_train, 
        y_train, 
        cv=3, 
        scoring='neg_mean_squared_error', 
        #Les scores sont négatifs, on prendra la racine carrée de leur moyenne négative pour obtenir le RMSE,
        #ce comportement est imposé par la convention de scikit-learn pour les métriques à minimiser
        n_jobs=-1
    )
    return np.sqrt(-scores.mean())

### **Lancement de l'optimisation**

Nous définissons des bornes (bounds) pour limiter l'espace de recherche et empêcher l'algorithme de tester des valeurs négatives (qui sont invalides pour le SVR). Ensuite, nous exécutons l'algorithme `L-BFGS-B` de Scipy qui est particulièrement adapté aux problèmes d'optimisation sous contraintes de bornes.


In [ ]:
#Définition des bornes (bounds)
# L'ordre correspond au vecteur: (C, epsilon, gamma)
bounds = [(100.0, 500000.0), (1e-4, 0.5), (1e-3, 2.0)]

print("Lancement de l'optimisation Scipy (L-BFGS-B)...")

# 4. Optimisation
result = minimize(
    objective_svr, 
    x0, 
    method='L-BFGS-B', 
    bounds=bounds, 
    options={'disp': True, 'maxiter': 20} # Limitation des itérations pour le temps de calcul
)

print("\n--- Résultats Scipy ---")
if result.success:
    print("Optimisation réussie.")
else:
    print("L'optimisation s'est arrêtée (limite d'itérations ou autre). Le résultat reste exploitable.")

print(f"Paramètres optimisés : C={result.x[0]:.2f}, epsilon={result.x[1]:.5f}, gamma={result.x[2]:.5f}")
print(f"Nouveau RMSE estimé : {result.fun:.4f}")


### **Évaluation du modèle SVR affiné**

Nous ré-entraînons maintenant un modèle SVR avec ces hyperparamètres ultra-précis trouvés par Scipy, pour vérifier sur notre jeu de test si l'on gagne en performance par rapport au modèle initial.

On crée alors un second pipeline pour le nouveau SVR optimisé par Scipy, puis on entraîne ce nouveau modèle sur les mêmes données de test que l'ancien.

In [ ]:
#Création d'un nouveau pipeline avec les paramètres optimisés par Scipy
pipeline_svr_scipy = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=result.x[0], epsilon=result.x[1], gamma=result.x[2]))
])

# Entraînement sur tout le jeu d'entraînement
pipeline_svr_scipy.fit(x_train, y_train)

# Prédiction sur le jeu de test
y_pred_scipy = pipeline_svr_scipy.predict(x_test)


Afin de mesurer les performances du nouveau modèle SVR optimisé par Scipy, on calcule son coefficient de détermination R² ainsi que sa RMSE, et on compare avec le modèle SVR trouvé initialement via `GridSearchCV`.

In [ ]:
# Performances du modèle optimisé par Scipy
rmse_scipy = np.sqrt(mean_squared_error(y_test, y_pred_scipy))
r2_scipy = r2_score(y_test, y_pred_scipy)

# Anciennes performances pour comparaison (issues du GridSearchCV)
rmse_grid = np.sqrt(mean_squared_error(y_test, predictions['SVR']))
r2_grid = r2_score(y_test, predictions['SVR'])

print("--- Comparaison sur le jeu de TEST ---")
print(f"SVR (GridSearch) -> RMSE: {rmse_grid:.4f} | R²: {r2_grid:.4f}")
print(f"SVR (Scipy)      -> RMSE: {rmse_scipy:.4f} | R²: {r2_scipy:.4f}")